# TP — Prédire le mot suivant avec des bigrammes (n=2)

**Objectif :** construire un programme qui propose jusqu'à **3 mots** pour continuer une phrase, en se basant sur 13 phrases d'apprentissage.

**Étapes :** phrases → tokens → bigrammes → comptes et probabilités → prédiction.

**Avant de commencer :** dans Google Colab, ouvre le panneau **Fichiers** (à gauche) et ajoute `dataset_bigrammes.json`. Puis exécute les cellules dans l'ordre.


## Étape 1 — Charger les données

On lit le fichier JSON pour récupérer :
- `phrases_apprentissage` : les 13 phrases utilisées pour apprendre ;
- `saisies_a_tester` : les 6 textes à tester.


In [1]:
import json  # Bibliothèque standard pour lire les fichiers JSON.
import re    # Bibliothèque standard pour découper le texte en mots.

# Ouvre le fichier JSON chargé dans Google Colab.
with open("dataset_bigrammes.json", "r", encoding="utf-8") as fichier:
    donnees = json.load(fichier)  # Transforme le JSON en dictionnaire Python.

# Affiche le nombre de phrases et le nombre de tests.
print("Nombre de phrases :", len(donnees["phrases_apprentissage"]))
print("Nombre de saisies à tester :", len(donnees["saisies_a_tester"]))


Nombre de phrases : 13
Nombre de saisies à tester : 6


## Étape 2 — Tokeniser une phrase

**Tokeniser** signifie transformer le texte en liste de mots (*tokens*). Pour rendre les comparaisons faciles, on met le texte en minuscules et on enlève la ponctuation.


In [2]:
def tokeniser(texte):  # Crée une fonction qui reçoit une phrase.
    texte = texte.lower()  # Convertit les majuscules en minuscules.
    tokens = re.findall(r"\b\w+\b", texte)  # Extrait les mots sans ponctuation.
    return tokens  # Renvoie la liste des mots.

# Exemple pour vérifier que la fonction fonctionne.
print(tokeniser("Le guide prépare une visite."))


['le', 'guide', 'prépare', 'une', 'visite']


## Étape 3 — Créer et compter les bigrammes

Un **bigramme** est une paire de mots voisins, par exemple `(guide, prépare)`.

Le dictionnaire `comptes` stocke, pour chaque contexte, combien de fois un mot apparaît après lui. On ajoute `DEBUT` avant chaque phrase pour pouvoir prédire le premier mot quand la saisie est vide.


In [3]:
comptes = {}  # Dictionnaire vide pour conserver les comptes.

# Parcourt toutes les phrases du corpus.
for phrase in donnees["phrases_apprentissage"]:
    tokens = tokeniser(phrase["texte"])  # Découpe la phrase en mots.
    tokens = ["DEBUT"] + tokens  # Ajoute le marqueur de début de phrase.

    # Parcourt les indices jusqu'à l'avant-dernier mot.
    for i in range(len(tokens) - 1):
        contexte = tokens[i]  # Premier mot du bigramme.
        mot_suivant = tokens[i + 1]  # Deuxième mot du bigramme.

        # Si ce contexte est nouveau, on prépare un dictionnaire.
        if contexte not in comptes:
            comptes[contexte] = {}

        # Si ce mot suivant est nouveau, on commence son compte à zéro.
        if mot_suivant not in comptes[contexte]:
            comptes[contexte][mot_suivant] = 0

        # Ajoute 1 à chaque fois que ce bigramme apparaît.
        comptes[contexte][mot_suivant] += 1

# Affiche les mots observés après "prépare".
print("Mots qui suivent 'prépare' :", comptes["prépare"])


Mots qui suivent 'prépare' : {'une': 4, 'un': 2, 'le': 1}


## Étape 4 — Calculer les probabilités et le top-3

La probabilité d'un mot est :

**P(mot suivant | contexte) = nombre d'apparitions du mot après le contexte ÷ total des apparitions après le contexte**.

Exemple : `P(une | prépare) = 4 / 7 = 57,14 %`.

Pour `n = 2`, **seul le dernier mot de la saisie** sert de contexte.


In [4]:
def predire(saisie):  # Fonction qui renvoie le contexte et jusqu'à 3 prédictions.
    tokens = tokeniser(saisie)  # Convertit la saisie en tokens.

    # Si rien n'est saisi, on utilise le début d'une phrase.
    if len(tokens) == 0:
        contexte = "DEBUT"
    else:
        contexte = tokens[-1]  # Récupère uniquement le dernier mot saisi.

    # Si le mot n'a jamais été vu, on ne peut pas prédire.
    if contexte not in comptes:
        return contexte, []  # [] signifie "aucune prédiction".

    total = sum(comptes[contexte].values())  # Nombre total des mots suivants.
    resultats = []  # Prépare la liste (mot, probabilité).

    # Parcourt tous les mots observés après le contexte.
    for mot, nombre in comptes[contexte].items():
        probabilite = nombre / total  # Calcule la probabilité.
        resultats.append((mot, probabilite))  # Ajoute le résultat.

    # Trie par probabilité décroissante (-x[1]), puis alphabétiquement (x[0]).
    resultats.sort(key=lambda x: (-x[1], x[0]))

    return contexte, resultats[:3]  # Renvoie seulement les 3 meilleurs mots.


## Étape 5 — Tester les 6 saisies

Pour chaque texte fourni par l'enseignant, afficher :
1. le contexte retenu ;
2. le top-3 avec les probabilités ;
3. la meilleure prédiction (ou le message d'absence de prédiction).


In [5]:
# Parcourt les six saisies fournies dans le fichier JSON.
for saisie in donnees["saisies_a_tester"]:
    contexte, top3 = predire(saisie)  # Lance la prédiction.

    print("\nSaisie :", repr(saisie))  # Affiche le texte, même s'il est vide.
    print("Contexte retenu :", contexte)  # Affiche le contexte choisi.

    if not top3:  # Vérifie si aucun mot n'a été prédit.
        print("Aucune prédiction : contexte non observé.")
    else:
        # Parcourt les meilleures prédictions disponibles.
        for mot, probabilite in top3:
            print(f"  {mot} : {probabilite:.4f} ({probabilite:.2%})")

        print("Meilleure prédiction :", top3[0][0])  # Première de la liste triée.



Saisie : 'le guide prépare'
Contexte retenu : prépare
  une : 0.5714 (57.14%)
  un : 0.2857 (28.57%)
  le : 0.1429 (14.29%)
Meilleure prédiction : une

Saisie : 'la classe prépare'
Contexte retenu : prépare
  une : 0.5714 (57.14%)
  un : 0.2857 (28.57%)
  le : 0.1429 (14.29%)
Meilleure prédiction : une

Saisie : 'le musée propose'
Contexte retenu : propose
  un : 0.5000 (50.00%)
  une : 0.5000 (50.00%)
Meilleure prédiction : un

Saisie : 'le musée'
Contexte retenu : musée
  propose : 0.5000 (50.00%)
  pour : 0.2500 (25.00%)
  puis : 0.2500 (25.00%)
Meilleure prédiction : propose

Saisie : 'astronaute'
Contexte retenu : astronaute
Aucune prédiction : contexte non observé.

Saisie : ''
Contexte retenu : DEBUT
  le : 0.6154 (61.54%)
  la : 0.3077 (30.77%)
  les : 0.0769 (7.69%)
Meilleure prédiction : le


## Interprétation (courte)

- **`le guide prépare`** et **`la classe prépare`** donnent la même réponse : leur dernier mot est `prépare`.
- **`le musée propose`** donne deux propositions (`un` et `une`) avec **50 %** chacune ; `un` est choisi en premier grâce au tri alphabétique.
- **`astronaute`** ne donne aucun résultat : ce contexte n'apparaît pas dans le corpus.
- **Une saisie vide** utilise `DEBUT` : le modèle prévoit alors le premier mot d'une phrase.

**Conclusion :** le modèle de bigrammes est simple à comprendre, mais il ne tient pas compte du sens ni des mots avant le dernier. Les tests montrent son fonctionnement, pas une évaluation de performance.
